In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("/kaggle/input/diabetes-risk/full_diabetes_dataset.csv")

# separate features and target
identifier_cols = ['County', 'State']
target_col = 'deaths_with_underlying_cause_diabetes_per_1k'

feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

X = df[feature_cols]
y = df[target_col]

# handle missing values
X = X.fillna(X.median())
y = y.fillna(y.median())

# split data (80% train, 10% val, 10% test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111
)

print("Train", X_train.shape[0])
print("Validation", {X_val.shape[0])
print("Test set", X_test.shape[0])

# get corresponding county/state info for each split
train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

train_identifiers = df.loc[train_idx, identifier_cols]
val_identifiers = df.loc[val_idx, identifier_cols]
test_identifiers = df.loc[test_idx, identifier_cols]

# standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

train_df = pd.concat([train_identifiers.reset_index(drop=True), 
                      pd.DataFrame(X_train.values, columns=feature_cols),
                      y_train.reset_index(drop=True).to_frame('target')], axis=1)
val_df = pd.concat([val_identifiers.reset_index(drop=True), 
                    pd.DataFrame(X_val.values, columns=feature_cols),
                    y_val.reset_index(drop=True).to_frame('target')], axis=1)
test_df = pd.concat([test_identifiers.reset_index(drop=True), 
                     pd.DataFrame(X_test.values, columns=feature_cols),
                     y_test.reset_index(drop=True).to_frame('target')], axis=1)

train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)